In [ ]:
# Multinomial Naive Bayes tanpa menggunakan hasil dari cosine similarity

from sklearn.model_selection import cross_val_predict, KFold
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from google.colab import drive
import pandas as pd

drive.mount('/content/gdrive')

# Memuat dataset yang telah dipreprocess
file_path = '/content/gdrive/MyDrive/modelling/preprocessed_data.csv'
df = pd.read_csv(file_path)

# Memastikan dataset yang telah dipreprocess tidak memiliki nilai NaN
df['comment'] = df['comment'].fillna('')

# Membuat vektor fitur menggunakan TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(stop_words=['indonesian'])
tfidf_matrix = tfidf_vectorizer.fit_transform(df['comment'])

# Memisahkan dataset menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(tfidf_matrix, df['label'], test_size=0.2, random_state=42)

# Melatih model Multinomial Naive Bayes
model_nb = MultinomialNB()
model_nb.fit(X_train, y_train)

# Melakukan prediksi pada data uji
y_pred = model_nb.predict(X_test)

# Menampilkan hasil evaluasi menggunakan 7 k-fold cross-validation
kf = KFold(n_splits=7, shuffle=True, random_state=42)
cross_val_results = cross_val_predict(model_nb, tfidf_matrix, df['label'], cv=kf)

# Menampilkan hasil evaluasi menggunakan 7 k-fold cross-validation
print("\n10 K-Fold Cross Validation Results:")
print("Accuracy:", accuracy_score(df['label'], cross_val_results))
print("\nClassification Report:\n", classification_report(df['label'], cross_val_results))
print("\nConfusion Matrix:\n", confusion_matrix(df['label'], cross_val_results))


Mounted at /content/gdrive

10 K-Fold Cross Validation Results:
Accuracy: 0.7956551255940258

Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.91      0.84       859
           1       0.83      0.64      0.72       614

    accuracy                           0.80      1473
   macro avg       0.80      0.77      0.78      1473
weighted avg       0.80      0.80      0.79      1473


Confusion Matrix:
 [[778  81]
 [220 394]]


In [ ]:
# Multinomial Naive Bayes menggunakan hasil dari cosine similarity

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/gdrive')

# Memuat dataset yang telah dipreprocess
file_path = '/content/gdrive/MyDrive/modelling/preprocessed_data.csv'
df = pd.read_csv(file_path)

# Memastikan dataset yang telah dipreprocess tidak memiliki nilai NaN
df['comment'] = df['comment'].fillna('')

# Membuat vektor fitur menggunakan TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(stop_words=['indonesian'])
tfidf_matrix = tfidf_vectorizer.fit_transform(df['comment'])

# Memisahkan dataset menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(tfidf_matrix, df['label'], test_size=0.2, random_state=42)

# Melatih model Multinomial Naive Bayes
model_nb = MultinomialNB()
model_nb.fit(X_train, y_train)

# Menggunakan cosine similarity pada matriks TF-IDF
keywords = ['atur', 'konten', 'kecewa']
tfidf_keywords = tfidf_vectorizer.transform(keywords)
cosine_similarities = cosine_similarity(tfidf_keywords, tfidf_matrix)

# Menghitung bobot akhir dengan cosine similarity sebagai pembobot
weighted_tfidf_matrix = cosine_similarities.dot(tfidf_matrix.toarray())

# Mendefinisikan fungsi cosine similarity
def calculate_cosine_similarity(tfidf_matrix, keyword_vector):
    cosine_similarity_scores = cosine_similarity(tfidf_matrix, keyword_vector)
    return cosine_similarity_scores[:, 1]

# Membuat matriks vektor kata kunci
keyword_vectors = np.zeros((len(keywords), tfidf_matrix.shape[1]))

for i, keyword in enumerate(keywords):
    keyword_index = tfidf_vectorizer.vocabulary_.get(keyword, -1)

    if keyword_index != -1:
        keyword_vector = np.zeros((tfidf_matrix.shape[1],))
        keyword_vector[keyword_index] = 1
        keyword_vectors[i, :] = keyword_vector


# Menghitung cosine similarity untuk setiap kata kunci
cosine_similarity_scores = cosine_similarity(tfidf_matrix, keyword_vectors)

# Menggabungkan hasil cosine similarity dengan matriks TF-IDF
weighted_tfidf_matrix = np.hstack([tfidf_matrix.toarray(), cosine_similarity_scores])

# Memisahkan dataset dengan bobot akhir menjadi data latih dan data uji
X_train_weighted, X_test_weighted, _, _ = train_test_split(weighted_tfidf_matrix, df['label'], test_size=0.2, random_state=42)

# Melatih model Multinomial Naive Bayes dengan bobot akhir
model_nb_weighted = MultinomialNB()
model_nb_weighted.fit(X_train_weighted, y_train)

# Melakukan prediksi pada data uji
y_pred_weighted = model_nb_weighted.predict(X_test_weighted)

# Menampilkan hasil evaluasi
print("Accuracy:", accuracy_score(y_test, y_pred_weighted))
print("\nClassification Report:\n", classification_report(y_test, y_pred_weighted))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_weighted))


Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
Accuracy: 0.8067796610169492

Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.93      0.85       170
           1       0.87      0.64      0.74       125

    accuracy                           0.81       295
   macro avg       0.82      0.78      0.79       295
weighted avg       0.82      0.81      0.80       295


Confusion Matrix:
 [[158  12]
 [ 45  80]]


In [ ]:
# Menampilkan komentar-komentar positif dan negatif
positive_comments = [comment for comment, label in zip(df.loc[y_test.index, 'comment'], y_pred_weighted) if label == 1]
negative_comments = [comment for comment, label in zip(df.loc[y_test.index, 'comment'], y_pred_weighted) if label == 0]

# Menampilkan komentar-komentar positif
print("\nKomentar Positif:")
for comment in positive_comments[:5]:  # Menampilkan 5 komentar pertama, sesuaikan sesuai kebutuhan
    print(comment)

# Menampilkan komentar-komentar negatif
print("\nKomentar Negatif:")
for comment in negative_comments[:5]:  # Menampilkan 5 komentar pertama, sesuaikan sesuai kebutuhan
    print(comment)



Komentar Positif:
didik bagus
terimakasih ytkids anak aku senang sekali nontonyavideonya bagussukses selalu
sangat muas
sukses
bagus sekali buat kembang anak

Komentar Negatif:
kalian biasa biasa sama ginian asal kalian tau walaupun konten kalian bukan buat anak anak video kamu golong kid friendly jadi set made kids suka kalau set kids otomatis kalian hilang komentarbellplaylist channelmu kalau tengah made kids channelmu denda ribu dolar lebih kalau tidak akan hapus akun sama penjara
wah beberes pilih tidak anak deh kek tarik cara belok target tonton deklarasi channel video siyap setting ulang
jadi wajib content select audio nya apakah vlog masakankulinerdan vlog keluarga harus di ubah atur audio nya
betul bijak nya rugi buat milik channel anak
terlalu lambat unduh
